# Лабораторная работа №1 - NumPy для нейронных сетей: от batch к forward pass

**Курс:** Нейронные сети  
**Максимальный балл:** 10  
**Центральный вопрос:** *из каких тензорных операций собирается прямой проход простого нейронного слоя и почему его удобно вычислять сразу для целого batch?*

В этой работе мы не изучаем NumPy «вообще». Нужные операции будут появляться ровно там, где они используются в нейронной сети: формы массивов → матричное умножение → broadcasting → активации → softmax → loss → векторизация.


## Паспорт работы

| Параметр | Значение |
|---|---|
| Тема | NumPy-операции, из которых строится forward pass |
| Максимальный балл | 10 |
| Ожидаемое время самостоятельной работы | 1.5-2 часа |
| Reference runtime | секунды на CPU |
| Оборудование | CPU, GPU не требуется |
| Внешние данные | не требуются |
| Основная библиотека | NumPy |
| Seed | 42 |
| Основной артефакт | заполненный notebook; отдельный Word-отчёт не нужен |

### Результаты обучения
После выполнения работы вы должны уметь:

- читать и прогнозировать формы массивов в batch-вычислениях;
- отличать поэлементные операции от матричного умножения;
- объяснять, как bias добавляется к batch с помощью broadcasting;
- реализовать ReLU, sigmoid и устойчивый softmax средствами NumPy;
- собрать прямой проход маленькой двухслойной сети и вычислить cross-entropy;
- сравнить Python-цикл с векторизованной реализацией и проверить их эквивалентность.


## Правила выполнения

- Можно пользоваться документацией, конспектами, примерами библиотек и ИИ-инструментами.
- Вы отвечаете за **корректность, воспроизводимость и понимание** всего кода, который сдаёте.
- Не меняйте выданные формы и seed без явной причины: они нужны для одинаковых контрольных проверок.
- Там, где требуется векторизация, Python-циклы использовать нельзя.
- Перед защитой notebook должен запускаться сверху вниз после заполнения всех `TODO`.


# 0. Окружение и воспроизводимость

Запустите этот раздел первым. Для работы достаточно Python и NumPy.


In [1]:
SEED = 42

import os
import sys
import time
import platform
import importlib.metadata as md
import numpy as np

os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", md.version("numpy"))
print("SEED:", SEED)


Python: 3.12.6
Platform: Windows-11-10.0.26100-SP0
NumPy: 2.5.3
SEED: 42


# 1. Цель и центральный вопрос

Один линейный слой для **одного** объекта можно записать как

$$z = xW + b.$$

Для целого batch формула не меняется:

$$Z = XW + b,$$

но теперь `X` - матрица объектов, а результат `Z` содержит выход слоя сразу для всех объектов.

Если

- $X \in \mathbb{R}^{B \times D}$ - batch из `B` объектов с `D` признаками;
- $W \in \mathbb{R}^{D \times H}$ - веса слоя;
- $b \in \mathbb{R}^{H}$ - bias,

то

$$XW \in \mathbb{R}^{B \times H}, \qquad XW+b \in \mathbb{R}^{B \times H}.$$

> **Центральный вопрос:** из каких тензорных операций собирается прямой проход простого нейронного слоя и почему его удобно вычислять сразу для целого batch?

К этому вопросу вернёмся в выводе.


# 2. Короткая подводка: NumPy-кирпичики

Это не отдельная теория по NumPy, а несколько операций, которые дальше будут использоваться непосредственно в нейросетевом вычислении.

## 2.1. `shape` и оси

Для матрицы формы `(3, 4)` первая ось отвечает строкам, вторая - столбцам. При агрегировании параметр `axis` задаёт ось, **по которой выполняется операция**.


In [2]:
A = np.arange(12).reshape(3, 4)
print("A =\n", A)
print("shape:", A.shape)
print("sum(axis=0):", A.sum(axis=0), "shape =", A.sum(axis=0).shape)
print("sum(axis=1):", A.sum(axis=1), "shape =", A.sum(axis=1).shape)


A =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape: (3, 4)
sum(axis=0): [12 15 18 21] shape = (4,)
sum(axis=1): [ 6 22 38] shape = (3,)


В дальнейшем строки почти всегда будут объектами batch. Поэтому, например, `sum(axis=1)` означает «получить одно значение для каждого объекта».

## 2.2. `*` и `@` - это разные операции

`*` умножает элементы поэлементно (с учётом broadcasting), а `@` выполняет матричное/скалярное произведение.


In [3]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print("a * b =", a * b)
print("a @ b =", a @ b)
print("Проверка dot product:", np.allclose(a @ b, np.sum(a * b)))


a * b = [ 4. 10. 18.]
a @ b = 32.0
Проверка dot product: True


Для матриц действует правило согласования внутренних размеров:

$$(B, D) @ (D, H) \rightarrow (B, H).$$

Именно эта операция лежит в основе линейного слоя.


In [4]:
X_demo = np.arange(12, dtype=float).reshape(3, 4)
W_demo = np.arange(8, dtype=float).reshape(4, 2)

print("X_demo:", X_demo.shape)
print("W_demo:", W_demo.shape)
print("X_demo @ W_demo:", (X_demo @ W_demo).shape)


X_demo: (3, 4)
W_demo: (4, 2)
X_demo @ W_demo: (3, 2)


## 2.3. Broadcasting

Bias слоя имеет одно значение на каждый выходной признак. Для batch его не нужно вручную копировать `B` раз: NumPy умеет добавить вектор формы `(H,)` к каждой строке матрицы формы `(B, H)`.


In [5]:
scores_demo = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])
bias_demo = np.array([0.1, -0.2, 0.5])

print("scores_demo shape:", scores_demo.shape)
print("bias_demo shape:", bias_demo.shape)
print("scores_demo + bias_demo =\n", scores_demo + bias_demo)


scores_demo shape: (2, 3)
bias_demo shape: (3,)
scores_demo + bias_demo =
 [[1.1 1.8 3.5]
 [4.1 4.8 6.5]]


**Мини-вопрос перед основной частью.** Не запуская код, попробуйте ответить: почему bias формы `(H,)` можно прибавить к `(B, H)`, а bias формы `(B,)` в общем случае нельзя?


> Ответ студента: Размеры сравниваются справа. (H,) совместим с (B, H), а (B,) обычно нет.

---

# 3. Основное задание

Все пять частей используют одни и те же размеры. Здесь `B` - размер batch, `D` - число входных признаков, `H` - число скрытых признаков, `C` - число классов.


In [6]:
B, D, H, C = 8, 4, 5, 3
rng = np.random.default_rng(SEED)

X = rng.normal(size=(B, D))
W1 = rng.normal(scale=0.5, size=(D, H))
b1 = rng.normal(scale=0.1, size=(H,))
W2 = rng.normal(scale=0.5, size=(H, C))
b2 = rng.normal(scale=0.1, size=(C,))

y = np.array([0, 2, 1, 1, 0, 2, 0, 1])

print("X ", X.shape)
print("W1", W1.shape, "b1", b1.shape)
print("W2", W2.shape, "b2", b2.shape)
print("y ", y.shape)


X  (8, 4)
W1 (4, 5) b1 (5,)
W2 (5, 3) b2 (3,)
y  (8,)


## Часть 1. Линейный слой: `X @ W + b`

Сначала **до запуска реализации** запишите ожидаемые формы:

| Выражение | Ожидаемая форма |
|---|---|
| `X @ W1` | `(8, 5)` |
| `X @ W1 + b1` | `(8, 5)` |
| `(X @ W1 + b1) @ W2` | `(8, 3)` |

Затем реализуйте `linear_batch(X, W, b)` **без Python-циклов**.


In [7]:
def linear_batch(X, W, b):
    # Линейный слой для batch.
    return X @ W + b


Z1 = linear_batch(X, W1, b1)
Z1


array([[ 0.06855132,  0.67916528, -0.21592587, -0.06034017,  0.2965551 ],
       [ 0.82310875,  1.20618287, -1.07475744, -1.68088261, -0.31411686],
       [ 0.01126421,  0.69263623, -0.2521161 , -0.18179151,  0.37044757],
       [-0.88778994, -0.76095257,  0.33822777,  0.31919941,  0.20732664],
       [-0.13524503,  0.29830696, -0.1958225 , -0.12324356,  0.0580607 ],
       [-0.23640731,  0.41341499, -0.25937176, -0.3099527 ,  0.28199571],
       [-0.02436067,  0.47332065, -0.25010095, -0.32550181,  0.24854283],
       [-1.19048368, -0.26686077,  0.33144202,  0.50462453,  0.87083616]])

**Короткая интерпретация.** В 1-2 предложениях объясните, что означает одна строка `Z1` и почему `b1` автоматически добавляется к каждой строке результата.

> Ответ студента:

> Одна строка Z1 содержит пять выходов для одного объекта. Bias добавляется к каждой строке через broadcasting.


In [8]:
assert Z1.shape == (B, H)
assert np.allclose(Z1[0], X[0] @ W1 + b1)
print("Часть 1: OK - форма Z1 =", Z1.shape)


Часть 1: OK - форма Z1 = (8, 5)


## Часть 2. Функции активации

Реализуйте две функции **без циклов**:

$$\mathrm{ReLU}(x)=\max(0,x),$$

$$\sigma(x)=\frac{1}{1+e^{-x}}.$$

Функции должны работать и со скаляром, и с вектором, и с матрицей NumPy благодаря поэлементным операциям.


In [9]:
def relu(x):
    # Обнуление отрицательных значений.
    return np.maximum(0, x)


def sigmoid(x):
    # Поэлементная sigmoid.
    return 1 / (1 + np.exp(-x))


A1 = relu(Z1)
print("A1 shape:", A1.shape)


A1 shape: (8, 5)


In [10]:
test_values = np.array([-2.0, 0.0, 2.0])

assert np.allclose(relu(test_values), [0.0, 0.0, 2.0])
assert np.all((sigmoid(test_values) > 0) & (sigmoid(test_values) < 1))
assert A1.shape == Z1.shape
assert np.all(A1 >= 0)

print("ReLU:", relu(test_values))
print("sigmoid:", np.round(sigmoid(test_values), 4))
print("Часть 2: OK")


ReLU: [0. 0. 2.]
sigmoid: [0.1192 0.5    0.8808]
Часть 2: OK


**Вопрос.** Почему после ReLU форма массива не меняется, хотя значения меняются существенно?

> Ответ студента:

> ReLU меняет значения поэлементно, сохраняя форму массива.


## Часть 3. Softmax: из logits в вероятности

На выходе последнего линейного слоя находятся **logits** - произвольные вещественные числа. Для многоклассовой классификации преобразуем каждую строку logits в распределение вероятностей:

$$p_{i,c}=\frac{e^{z_{i,c}}}{\sum_j e^{z_{i,j}}}.$$

Для численной устойчивости перед экспонентой вычтем максимум **в каждой строке**. Реализуйте softmax для матрицы формы `(B, C)` без циклов.

Обратите внимание на `axis=1` и `keepdims=True`: максимум и сумма считаются отдельно для каждого объекта batch, а сохранённая размерность позволяет удобно использовать broadcasting.


In [11]:
def softmax(logits):
    # Максимум каждой строки для устойчивости.
    shifted = logits - logits.max(axis=1, keepdims=True)
    # Экспоненты сдвинутых logits.
    exp_logits = np.exp(shifted)
    # Нормировка по классам.
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


logits = linear_batch(A1, W2, b2)
probs = softmax(logits)
print("logits shape:", logits.shape)
print("probs shape:", probs.shape)


logits shape: (8, 3)
probs shape: (8, 3)


In [12]:
assert probs.shape == (B, C)
assert np.all(probs >= 0)
assert np.all(probs <= 1)
assert np.allclose(probs.sum(axis=1), 1.0)

large_logits = np.array([[1000.0, 1001.0, 1002.0]])
large_probs = softmax(large_logits)
assert np.all(np.isfinite(large_probs))
assert np.allclose(large_probs.sum(axis=1), 1.0)

print("Первые две строки probabilities:\n", np.round(probs[:2], 4))
print("Softmax для больших logits:", np.round(large_probs, 4))
print("Часть 3: OK")


Первые две строки probabilities:
 [[0.2377 0.3399 0.4225]
 [0.2124 0.2088 0.5788]]
Softmax для больших logits: [[0.09   0.2447 0.6652]]
Часть 3: OK


**Ответьте коротко:**

1. Почему здесь softmax считается по `axis=1`, а не по `axis=0`?
2. Почему вычитание максимума строки не меняет итоговые вероятности softmax?
3. Что должно быть верно для суммы элементов каждой строки `probs`?

> Ответ студента:

> 1. axis=1 соответствует классам одного объекта.
> 2. Множитель exp(-max) сокращается в дроби.
> 3. Сумма строки равна 1 с учётом погрешности.


## Часть 4. Собираем forward pass и cross-entropy

Теперь соединим операции в маленькую двухслойную сеть:

$$X \rightarrow \text{Linear}(D,H) \rightarrow \text{ReLU} \rightarrow \text{Linear}(H,C) \rightarrow \text{Softmax}.$$

Реализуйте функцию `forward_two_layer`. Она должна вернуть **и logits, и probabilities**, чтобы на защите было видно различие этих двух сущностей.

После этого вычислите среднюю cross-entropy по правильным классам:

$$L=-\frac{1}{B}\sum_{i=1}^{B}\log p_{i,y_i}.$$

Используйте индексирование NumPy, чтобы выбрать вероятность правильного класса для каждого объекта без цикла.


In [13]:
def forward_two_layer(X, W1, b1, W2, b2):
    # Linear, ReLU, linear, softmax.
    hidden = relu(linear_batch(X, W1, b1))
    logits = linear_batch(hidden, W2, b2)
    return logits, softmax(logits)


def cross_entropy(probs, y, eps=1e-12):
    # Вероятности правильных классов.
    p_true = probs[np.arange(len(y)), y]
    # Защита от log(0).
    p_true = np.clip(p_true, eps, 1.0)
    # Средняя cross-entropy.
    return -np.log(p_true).mean()


logits_net, probs_net = forward_two_layer(X, W1, b1, W2, b2)
loss = cross_entropy(probs_net, y)


In [14]:
assert logits_net.shape == (B, C)
assert probs_net.shape == (B, C)
assert np.allclose(probs_net.sum(axis=1), 1.0)
assert np.isfinite(loss) and loss >= 0

pred = probs_net.argmax(axis=1)
accuracy = np.mean(pred == y)

print("Predicted classes:", pred)
print("True classes:     ", y)
print(f"Accuracy на искусственном batch: {accuracy:.3f}")
print(f"Cross-entropy: {loss:.4f}")
print("Часть 4: OK")


Predicted classes: [2 2 2 2 2 2 2 2]
True classes:      [0 2 1 1 0 2 0 1]
Accuracy на искусственном batch: 0.250
Cross-entropy: 1.1508
Часть 4: OK


**Небольшой мысленный эксперимент.** Без запуска дополнительного кода ответьте:

- что произойдёт с вкладом одного объекта в cross-entropy, если вероятность его правильного класса приблизится к `1`?
- что произойдёт, если она приблизится к `0`?

> Ответ студента:

> При p → 1 ошибка стремится к 0. При p → 0 она растёт; clip ограничивает её значением -log(eps).


## Часть 5. Python-цикл против векторизации

Нейросетевой код обычно работает с batch не потому, что «так принято», а потому что операции над большим массивом можно передать оптимизированным численным библиотекам целиком.

1. Реализуйте `linear_loop`: цикл разрешён **по объектам batch**, но внутри строки используйте `@`.
2. Реализуйте `linear_vectorized` одной NumPy-формулой.
3. Проверьте совпадение результатов через `np.allclose`.
4. Измерьте время обеих версий на более крупном искусственном batch.

Не требуется получить конкретный коэффициент ускорения: он зависит от процессора, BLAS и текущей нагрузки системы. Важно корректное сравнение и интерпретация.


In [15]:
def linear_loop(X, W, b):
    # Линейный слой по одному объекту.
    out = np.empty((X.shape[0], W.shape[1]))
    for i in range(X.shape[0]):
        out[i] = X[i] @ W + b
    return out


def linear_vectorized(X, W, b):
    # Линейный слой одной формулой.
    return X @ W + b


In [16]:
def benchmark(fn, *args, repeats=3):
    """Минимальное время из нескольких запусков; этого достаточно для учебного сравнения."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return min(times)

rng_bench = np.random.default_rng(SEED + 1)
X_bench = rng_bench.normal(size=(20_000, 64))
W_bench = rng_bench.normal(size=(64, 32))
b_bench = rng_bench.normal(size=(32,))

out_loop = linear_loop(X_bench, W_bench, b_bench)
out_vec = linear_vectorized(X_bench, W_bench, b_bench)
assert np.allclose(out_loop, out_vec)

t_loop = benchmark(linear_loop, X_bench, W_bench, b_bench)
t_vec = benchmark(linear_vectorized, X_bench, W_bench, b_bench)

print(f"loop:       {t_loop:.6f} s")
print(f"vectorized: {t_vec:.6f} s")
if t_vec > 0:
    print(f"отношение loop/vectorized: {t_loop / t_vec:.2f}x")
print("Результаты совпадают:", np.allclose(out_loop, out_vec))


loop:       0.041772 s
vectorized: 0.002881 s
отношение loop/vectorized: 14.50x
Результаты совпадают: True


**Интерпретация benchmark.** Ответьте в 2-4 предложениях:

- какая версия оказалась быстрее на вашей машине;
- почему это сравнение нельзя превращать в обязательный порог вида «ускорение не менее 20×»;
- почему `np.allclose`, а не точное `==`, является разумной проверкой для численных вычислений.

> Ответ студента:

> Векторизация быстрее: цикл 0.041772 с, векторизация 0.002881 с. Ускорение зависит от CPU, BLAS и нагрузки. np.allclose учитывает численную погрешность.


# 4. Итоговая проверка результата

После заполнения всех `TODO` эта ячейка должна выполняться без ошибок. Она проверяет не конкретные случайные числа, а свойства результата: формы, допустимые диапазоны и согласованность разных реализаций.


In [17]:
# Shapes
assert X.shape == (B, D)
assert Z1.shape == (B, H)
assert logits_net.shape == (B, C)
assert probs_net.shape == (B, C)

# Activations
assert np.all(relu(np.array([-1.0, 0.0, 1.0])) == np.array([0.0, 0.0, 1.0]))
assert np.allclose(sigmoid(np.array([0.0])), np.array([0.5]))

# Softmax
assert np.all(np.isfinite(probs_net))
assert np.all(probs_net >= 0)
assert np.all(probs_net <= 1)
assert np.allclose(probs_net.sum(axis=1), 1.0)

# Loss
assert np.isfinite(loss)
assert loss >= 0

# Vectorization equivalence
assert np.allclose(
    linear_loop(X_bench[:100], W_bench, b_bench),
    linear_vectorized(X_bench[:100], W_bench, b_bench),
)

print("Все итоговые проверки пройдены.")


Все итоговые проверки пройдены.


# 5. Вывод

Ответьте коротко, но содержательно - примерно 5-8 предложений суммарно.

1. **Ответ на центральный вопрос:** из каких операций собрался forward pass этой сети?
2. Какие формы имели `X`, `W1`, `b1`, `Z1`, `W2` и финальные probabilities? Почему именно такие?
3. Где в работе использовался broadcasting?
4. Зачем softmax считать отдельно для каждой строки batch?
5. Что показало сравнение цикла и векторизованного решения?
6. Назовите одну типичную ошибку с формами/осями, которая могла бы изменить результат или сломать вычисление.

> **Вывод студента:**

> Прямой проход включает матричные умножения, bias, ReLU и softmax. Формы: X (8, 4), W1 (4, 5), b1 (5,), Z1 (8, 5), W2 (5, 3), probs (8, 3). Строки задают объекты, столбцы задают признаки или классы. Broadcasting применяется к bias и при нормировке softmax. Softmax даёт отдельное распределение классов для каждой строки. Цикл и векторизация дают близкие результаты. Типичная ошибка: axis=0 вместо axis=1 в softmax.


# 6. Источники и использованные инструменты

### Материалы курса
Подводка к работе использует адаптированные фрагменты материалов курса по NumPy:

- `01_vectors.ipynb` - векторы, поэлементные операции, dot product;
- `02_matrices.ipynb` - формы матриц и матричное умножение;
- `0_NumPy.ipynb` - `shape`, `axis`, агрегирование, поэлементные операции и `@`.

### Дополнительные источники студента
- Нет.

### ИИ-инструменты
- ChatGPT (Codex): код, пояснения и проверка запуска.

Историю промптов прикладывать не требуется. Указывайте только существенные внешние источники и инструменты, повлиявшие на решение.


# 7. Оценивание и подготовка к защите

| Компонент | Баллы | Что проверяется в ЛР1 |
|---|---:|---|
| Корректность реализации | 3 | `X @ W + b`, активации, softmax, forward pass, cross-entropy |
| Эксперимент и проверки | 2 | sanity checks, `np.allclose`, корректное сравнение цикла и векторизации |
| Анализ и выводы | 1 | осмысленный ответ на центральный вопрос и интерпретация |
| Устная защита | 4 | формы, оси, broadcasting, смысл операций, live modification |

**Hard gate:** если за устную часть получено меньше **2 из 4 баллов**, лабораторная не засчитывается независимо от качества notebook.

Перед защитой убедитесь, что можете без подсказки:

- назвать форму результата любой ключевой строки;
- объяснить разницу между `*` и `@`;
- объяснить, как работает broadcasting bias;
- объяснить `axis=1` и `keepdims=True` в softmax;
- показать, где появляются logits и где probabilities;
- объяснить cross-entropy через вероятность правильного класса;
- предсказать последствия небольшой правки кода до её запуска.


---

### Самопроверка перед сдачей

- [x] Все `TODO` заполнены.
- [x] Notebook запускается сверху вниз без ошибок.
- [x] Все итоговые `assert` проходят.
- [ ] Понимание форм и осей проверено самостоятельно.
- [x] В softmax probabilities суммируются до 1 **по классам для каждого объекта**.
- [x] Циклическая и векторизованная реализации дают одинаковый результат.
- [x] Вывод отвечает на центральный вопрос.
- [x] Существенные источники и ИИ-инструменты указаны.
- [ ] Готовность объяснить и изменить код проверена самостоятельно.

Пункты о понимании и защите требуют самостоятельной проверки.
